# Clasificadores de Cocinas 🍜

En esta lección vamos a usar el dataset limpio de la lección anterior
para **entrenar modelos de clasificación** que predecirán de qué cocina
es una receta basándose en sus ingredientes.

## 1. Cargar datos limpios

Usamos `cleaned_cuisines.csv` que creamos en la lección anterior.
Ya está balanceado con SMOTE y sin los ingredientes comunes.

In [ ]:
import pandas as pd

cuisines_df = pd.read_csv('../data/cleaned_cuisines.csv')
cuisines_df.head()

## 2. Importar modelos y métricas

- **LogisticRegression**: Nuestro primer clasificador
- **SVC**: Support Vector Classifier (para comparar después)
- **Métricas**: Para evaluar qué tan bien funciona el modelo

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report, precision_recall_curve
from sklearn.svm import SVC
import numpy as np

## 3. Separar features y labels

- **Labels** (y): La columna `cuisine` — lo que queremos predecir
- **Features** (X): Todo lo demás — los ingredientes (0 o 1)

In [ ]:
cuisines_label_df = cuisines_df['cuisine']
cuisines_label_df.head()

In [ ]:
cuisines_feature_df = cuisines_df.drop(['Unnamed: 0', 'cuisine'], axis=1)
cuisines_feature_df.head()

## 4. Dividir en train/test

Usamos 70% para entrenar y 30% para evaluar.
Esto es clave: **nunca evalúes el modelo con los mismos datos con los que lo entrenaste**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(cuisines_feature_df, cuisines_label_df, test_size=0.3)

## 5. Entrenar Logistic Regression

Aquí es donde la magia ocurre. El modelo aprende la relación entre
ingredientes y cocinas.

- `multi_class='ovr'`: One-vs-Rest — entrena un clasificador binario por cada cocina
- `solver='liblinear'`: Algoritmo de optimización para datasets pequeños

In [ ]:
lr = LogisticRegression(multi_class='ovr', solver='liblinear')
model = lr.fit(X_train, np.ravel(y_train))

accuracy = model.score(X_test, y_test)
print(f'Accuracy: {accuracy:.2%}')

## 6. Ver una predicción

Veamos qué ingredientes tiene la fila #50 del test set
y qué cocina predijo el modelo.

In [ ]:
print(f'Ingredientes: {X_test.iloc[50][X_test.iloc[50]!=0].keys().tolist()}')
print(f'Cocina real: {y_test.iloc[50]}')

## 7. Ver probabilidades

El modelo no solo predice la cocina — también calcula la **probabilidad**
para cada clase. Esto te dice qué tan seguro está.

In [ ]:
test = X_test.iloc[50].values.reshape(-1, 1).T
proba = model.predict_proba(test)
classes = model.classes_

resultdf = pd.DataFrame(data=proba, columns=classes)
resultdf.T.sort_values(by=[0], ascending=[False]).head()

## 8. Evaluar con classification report

El reporte te muestra para **cada cocina**:
- **Precision**: De los que predije como esta cocina, ¿cuántos lo eran?
- **Recall**: De los que ERAN esta cocina, ¿cuántos detecté?
- **F1-score**: Balance entre precision y recall

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

---

## 🧪 Experimentos para entender mejor

Ahora que el modelo está entrenado, hacé estos experimentos
para realmente entender qué está pasando.

### Experimento 1: Cambiar el solver

El `solver` es el algoritmo que el modelo usa para encontrar
los mejores coeficientes. Probá dos diferentes y compará:

In [ ]:
# Solver liblinear
lr_liblinear = LogisticRegression(multi_class='ovr', solver='liblinear')
model_liblinear = lr_liblinear.fit(X_train, np.ravel(y_train))
acc_liblinear = model_liblinear.score(X_test, y_test)

# Solver lbfgs
lr_lbfgs = LogisticRegression(multi_class='ovr', solver='lbfgs')
model_lbfgs = lr_lbfgs.fit(X_train, np.ravel(y_train))
acc_lbfgs = model_lbfgs.score(X_test, y_test)

print(f'liblinear: {acc_liblinear:.2%}')
print(f'lbfgs:     {acc_lbfgs:.2%}')

### Experimento 2: Probar otras filas

Cambiate el número `50` por otros (0, 100, 200, etc.)
y mirá si el modelo acierta o se equivoca.

In [ ]:
for i in [0, 100, 200, 500]:
    ingredientes = X_test.iloc[i][X_test.iloc[i]!=0].keys().tolist()
    real = y_test.iloc[i]
    pred = model.predict(X_test.iloc[i].values.reshape(1, -1))[0]
    status = '✅' if real == pred else '❌'
    print(f'{status} Fila {i}: {real} → predicho: {pred} | ingredientes: {ingredientes[:3]}...')

### Experimento 3: Confusion Matrix

La matriz de confusión te dice **dónde se equivoca** el modelo.
Las filas son las cocinas reales, las columnas son las predichas.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=model.classes_, columns=model.classes_)
cm_df

### Experimento 4: ¿Qué cocina es más fácil de predecir?

Mirá el classification report de arriba:
- ¿Qué cocina tiene mayor **recall**? (la más fácil de detectar)
- ¿Qué cocina tiene menor **recall**? (la que más se confunde)
- ¿Por qué creés que pasa eso?

---

## ✅ Resumen

1. **Cargamos** el dataset limpio de cocinas
2. **Separamos** features (ingredientes) y labels (cuisine)
3. **Dividimos** en train (70%) y test (30%)
4. **Entrenamos** Logistic Regression con solver liblinear
5. **Evaluamos** con accuracy, classification report y confusion matrix

### Conceptos clave

| Término | Significado |
|---------|-------------|
| **Solver** | Algoritmo de optimización |
| **multi_class** | Cómo maneja más de 2 clases |
| **OvR** | One-vs-Rest: un clasificador binario por clase |
| **Accuracy** | % general de aciertos |
| **Precision** | De los predichos como X, ¿cuántos son X? |
| **Recall** | De los que son X, ¿cuántos detectó? |
| **F1-score** | Balance precision-recall |